[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q4/00_orient_and_precommit.ipynb)

# R2-Q4 Week 1 — Orient and precommit your experimental design

### This notebook produces `precommit.json`, which holds the three decisions that drive every notebook after it.

This notebook walks you through R2-Q4's setup. You will read the question page, inspect PlantVillage to see which diseases appear on multiple hosts, and make three decisions that lock the experimental design before any classifier is trained. The decisions you make here drive every notebook after it — change one of them mid-experiment and the comparison falls apart.

By the end of this notebook you will have:

- A `precommit.json` file holding three decisions and the reasoning behind each: your disease list (which multi-host diseases qualify, which single-host diseases serve as controls), your hold-out scheme (fixed train/test host assignment, or leave-one-host-out per disease), and your cross-host accuracy aggregation (per-disease, or one overall number).
- A short orientation note in your own words, restating R2-Q4 as you understand it.
- Submitted EQ#1.

In [ ]:
# Cell 0 — probe
try:
    import irilab2026 as iri
    print(f"irilab2026 {iri.__version__} already installed.")
    print("If you want to pull the latest changes, run Cell 1 below.")
except ModuleNotFoundError:
    print("irilab2026 not installed — run Cell 1 below.")

In [ ]:
# Cell 1 — install
# First run in a fresh Colab session: run this cell.
# Later runs in the same session: skip this cell.
!pip install git+https://github.com/geraldmc/irilab2026.git@main

**A note about a possible restart prompt.** If Colab shows a "RESTART SESSION" banner after the install: click it, wait for the kernel to reconnect, then continue from the next cell. The install does not need to be run again. If no banner appears, ignore this and move on.

In [ ]:
# Cell 2 — mount and setup
# Mount Drive, set up irilab2026, point to the R2-Q4 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r2_q4")
print(f"Output directory: {OUTPUT_DIR}")

### 1) Orient to the question

R2-Q4 asks a question about what a model has actually learned. You train a classifier on PlantVillage, where most diseases are photographed on more than one host plant — late blight on both potato and tomato, bacterial spot on pepper and tomato, and so on. The classifier reaches high accuracy. But high accuracy on its own doesn't tell you *what* the model is keying on. It could be reading the disease — the lesions, the discoloration, the texture of the symptom on the leaf. Or it could be reading the host — the shape and color of a tomato leaf versus a potato leaf — and getting the disease right only because, in this particular dataset, each disease happens to sit on a predictable set of hosts. Both strategies score well on a normal test set. Only one of them is learning the thing you care about.

The way to tell them apart is to break the host–disease correlation on purpose. You train on the mixed-host data, then evaluate on a host the model never saw for that disease — one host withheld per disease. A model that learned the *symptom* should still recognize the disease on the new host, because the symptom travels. A model that learned the *host template* should fail, because the host it was leaning on has changed. The drop in accuracy from the within-host test to this cross-host test is the diagnostic: a large drop is the signature of a host-template shortcut. This is the same move R1-Q3 makes in Rationale 1 — checking whether a model got the right answer for the right reason by evaluating it on data designed to expose the wrong reason.

This kind of failure has a name in the machine-learning literature — a shortcut, where a model latches onto a feature that correlates with the label in the training data but isn't the feature you intended (Geirhos et al., 2020). A related caution comes from Noyan (2022), who showed a plant-disease classifier could reach roughly half its accuracy from the image background alone, with the leaf masked out — a reminder that "the model is reading something other than the symptom" is not a hypothetical.

Two cautions to carry from the very start, because they shape how you're allowed to phrase your conclusions later:

- **Conclusions are disease-specific.** If rust fails to transfer from one host to another, the finding is "this model failed to transfer rust across these hosts" — not "CNNs can't learn disease features." Each disease is its own small experiment. Resist the urge to generalize across diseases or to models in general.
- **A shared disease *name* is not a shared *pathogen*.** Two hosts labeled with the same disease may be infected by genuinely different organisms that happen to share a common name. So if a disease *does* transfer cleanly across hosts, that's ambiguous: it could mean the model learned a generalizable symptom, or it could mean the two host–pathogen pairs simply look alike. Both readings are legitimate and both belong in your write-up. Successful transfer is weaker evidence than failed transfer.

**Your turn.** Before the precommits, write a short orientation note in your own words — three or four sentences restating what R2-Q4 is asking and why withholding a host is the way to answer it. This is the basis for your EQ#1 submission. If you can't state the diagnostic plainly, the decisions in the sections below won't have a firm footing.

### 2) Inspect PlantVillage: identify multi-host diseases and per-host sample counts

The experiment only works on diseases that appear on **more than one host**. Those are the only diseases where "withhold a host" is even a meaningful move — you need at least one host to train on and a different host to test on. A disease that appears on a single host can't be transfer-tested at all, though it can still earn its place in the experiment as a control (a class the model learns but that never gets a cross-host evaluation).

So before locking a disease list, you need two things from the data: which diseases are multi-host, and how many images each host contributes to each disease. The second number matters as much as the first — a disease that technically appears on two hosts, but where one host has only a handful of images, can't anchor a stable held-out test on that host. You're looking for diseases that are multi-host *and* well-sampled on the hosts you'd hold out.

Note: the cell below loads the `"full"` PlantVillage variant. The disease-list decision needs real counts, not the 50-images-per-class `"tiny"` subset, and the count works off the metadata table, not the images themselves. The first call downloads about 1.5 GB and caches it to Drive; NB01 needs that same download next week, so this front-loads it rather than adding cost.

In [ ]:
### 2.1) Load PlantVillage and build the host × disease table
import pandas as pd
import irilab2026 as iri

# metadata: one row per image, with `host` and `disease` already parsed out of
# the class label (e.g. "Potato___Late_blight" -> host="Potato",
# disease="Late_blight"). images: the PIL images, not needed in this notebook.
metadata, images = iri.load_plantvillage(variant="full")

# How many images of each disease on each host.
host_disease = (
    metadata.groupby(["disease", "host"])
    .size()
    .rename("n_images")
    .reset_index()
)

# How many distinct hosts each disease appears on.
hosts_per_disease = (
    host_disease.groupby("disease")["host"]
    .nunique()
    .rename("n_hosts")
    .sort_values(ascending=False)
)

print("Distinct hosts per disease:")
print(hosts_per_disease.to_string())

In [ ]:
### 2.2) Per-host image counts for the multi-host diseases
# These are the only diseases the cross-host experiment can run on. Read the
# per-host counts here against whatever per-host minimum you'll set in the
# disease-list precommit below — a host with too few images can't anchor a
# held-out test.
multi_host = hosts_per_disease[hosts_per_disease >= 2].index

view = (
    host_disease[host_disease["disease"].isin(multi_host)]
    .sort_values(["disease", "n_images"], ascending=[True, False])
    .reset_index(drop=True)
)
print(view.to_string())

Two things to watch for in what you just printed:

- **`healthy` is multi-host but isn't a disease.** Every crop has a "healthy" class, so `healthy` shows up with a high host count. It isn't a disease to transfer-test. Whether it belongs in your experiment at all — as a class the model still has to tell apart from the diseases — is a decision you make in the next section, not something to read off the table.
- **A shared disease name can hide very different-looking cases.** A disease that appears on two visually distinct hosts (the kind of thing the orientation warned about) is exactly the boundary case worth flagging now. It might be your most informative test — or a confound you'd rather drop. Either way, name it deliberately in the next section rather than letting the count decide for you.

### 3) Precommit 1: disease list (which multi-host diseases to include; which single-host diseases to keep as controls)

This is the first of three decisions you lock in this notebook. From Section 2 you now know which diseases are multi-host and how many images each host contributes. Lock the list: which multi-host diseases are the experiment's transfer subjects, and which single-host diseases (if any) you keep as controls.

The question page suggests about four diseases. That's a guide, not a quota — the right number falls out of your own counts. Three considerations drive the cut:

- **Host count.** A transfer subject needs at least two hosts. One to train on, a different one to hold out.
- **Sample minimum per host.** Set a floor — `min_images_per_host` below. A multi-host disease only qualifies if *every* host you intend to use clears it. A disease on two hosts where one host has 30 images can't give you a stable cross-host estimate on that host.
- **Boundary cases.** Diseases whose two hosts look very different, or where the shared name might mask different pathogens, are judgment calls. Keep them as your sharpest tests, or drop them as confounds — but decide on purpose.

A word on the default list in the cell below, because it comes with a warning. The diseases pre-filled there are PlantVillage's *usual* multi-host candidates, written from general knowledge of the dataset's class structure — **not** read off the `groupby` you just ran. They are a worked example of the format and a starting point, nothing more. Confirm every entry against your own Section 2 output before you submit: check that each disease really does appear on the hosts you think it does, and that each of those hosts clears your `min_images_per_host` floor. If your counts disagree with the default, your counts win.

In [ ]:
### 3.1) Initialize the precommit dict and commit Precommit 1
# The running precommit dict. Each of the three precommit sections adds one
# top-level field; the closeout (Section 6) writes the whole thing to
# precommit.json in a single json.dump.
precommit = {
    "disease_list": None,            # Precommit 1 — this section
    "hold_out_scheme": None,         # Precommit 2
    "cross_host_aggregation": None,  # Precommit 3
}

# Commit Precommit 1: the disease list.
#
# PROVISIONAL DEFAULT. The four diseases below are PlantVillage's usual
# multi-host candidates, written from general knowledge of the dataset, NOT
# from your Section 2 groupby. Confirm each one against `host_disease` before
# you submit: that the disease appears on the hosts you expect, and that each
# host clears `min_images_per_host`. If your counts disagree, edit the list.
precommit["disease_list"] = {
    "multi_host_diseases": [
        "Early_blight",    # typically Potato + Tomato
        "Late_blight",     # typically Potato + Tomato
        "Bacterial_spot",  # typically Pepper_bell + Tomato (+ Peach in some builds)
        "Powdery_mildew",  # typically Cherry + Squash
    ],
    "control_single_host_diseases": [],  # single-host diseases kept as controls; [] = none
    "include_healthy": False,            # whether "healthy" is a class in the experiment
    "min_images_per_host": 100,          # every host of a kept disease must clear this
    "reasoning": (
        # Replace this placeholder with your own reasoning before submission.
        # Cover at minimum: which multi-host diseases you kept and which you
        # dropped, citing the Section 2 per-host counts that justify each cut;
        # your min_images_per_host floor and why you set it there; how you
        # handled "healthy"; and which single-host diseases (if any) you kept
        # as controls and what they control for.
        "PLACEHOLDER — REWRITE BEFORE SUBMISSION."
    ),
}

# Print back what just got committed.
dl = precommit["disease_list"]
print("Multi-host diseases (transfer subjects):")
for d in dl["multi_host_diseases"]:
    print(f"  - {d}")
print(f"Control single-host diseases: {dl['control_single_host_diseases'] or 'none'}")
print(f"Include healthy:     {dl['include_healthy']}")
print(f"Min images per host: {dl['min_images_per_host']}")

### 4) Precommit 2: hold-out scheme (leave-one-host-out per disease vs fixed train/test host assignment)

You've decided *which* diseases. Now decide *how* you withhold a host for each one. There are two schemes, and they answer slightly different questions, so you lock the choice before any training — not after seeing which one gives a nicer result.

- **Fixed host assignment** — for each disease, you pick one host to be the held-out test host; every other host of that disease goes into training. This trains a **single** classifier (the `classifier.pkl` NB01 produces), and each disease gets one cross-host number. Simpler, cheaper, and what NB01 through NB03 are built around. The cost: your result depends on *which* host you held out, so the choice of held-out host is itself part of your method and has to be justified.
- **Leave-one-host-out** — for each disease, you rotate the held-out host: train on all-but-one, test on the one, then repeat with a different host held out. Every host gets a turn as the target, which is more thorough and gives you a per-host view of transfer. The cost is real: the training set changes each time a different host is withheld, so this needs **one classifier per held-out configuration**, and NB01 is not pre-built to loop over them.

The default below is fixed host assignment, for the same reason R2-Q3 defaulted to its simpler design: it matches the scaffold (one trained classifier) and keeps the Week-2 training budget to a single run. Choose leave-one-host-out only if a per-host view is central to your question, and know that it changes NB01's shape.

If you go with fixed host assignment, here's a sensible rule for *which* host to hold out: hold out a host that is well-sampled enough to give a stable test estimate. Holding out a host with only a handful of images means your cross-host accuracy for that disease rests on those few images, and a single number computed on, say, 25 images is noisy. The default assignment below holds out the best-sampled host for each disease for exactly this reason — but it presumes the default disease list above, so if you edited that list, edit this to match. The two have to stay in sync.

(This scheme choice also shapes what NB02 can report: under leave-one-host-out each disease produces several held-out results, which raises a "roll them up or keep them separate" question downstream. That's an NB02 decision; it's flagged here only so you know the scheme you pick now has a consequence three notebooks out.)

In [ ]:
### 4.1) Commit Precommit 2: the hold-out scheme
# Default is "fixed_host" — one held-out test host per disease, the rest train,
# a single classifier. "leave_one_host_out" rotates the held-out host and needs
# one classifier per configuration; NB01 is not pre-built for it.
precommit["hold_out_scheme"] = {
    "scheme": "fixed_host",   # one of: "fixed_host", "leave_one_host_out"
    # For "fixed_host": which host to withhold for each disease's cross-host test.
    # PROVISIONAL — holds out the best-sampled host per disease; confirm each
    # held-out host has enough images to anchor a stable test, and keep these
    # keys in sync with the disease list above. For "leave_one_host_out" this
    # dict is ignored (every host is held out in turn).
    "held_out_host": {
        "Early_blight": "Tomato",
        "Late_blight": "Tomato",
        "Bacterial_spot": "Tomato",
        "Powdery_mildew": "Squash",
    },
    "reasoning": (
        # Replace this placeholder with your own reasoning before submission.
        # Cover at minimum: which scheme and why; if "fixed_host", how you chose
        # the held-out host for each disease and what that single choice can and
        # cannot claim about transfer; if "leave_one_host_out", your awareness
        # that NB01 must train one classifier per held-out configuration.
        "PLACEHOLDER — REWRITE BEFORE SUBMISSION."
    ),
}

# Print back what just got committed.
ho = precommit["hold_out_scheme"]
print(f"Hold-out scheme: {ho['scheme']}")
if ho["scheme"] == "fixed_host":
    print("Held-out test host per disease:")
    for disease, host in ho["held_out_host"].items():
        print(f"  - {disease}: hold out {host}")
else:
    print("  (every host held out in turn; one classifier per configuration)")

### 5) Precommit 3: cross-host accuracy aggregation (per-disease vs overall)

The last decision is how you report the cross-host accuracy once you have it. You can report it **per-disease** — one transfer number for each disease — or **overall** — pool every cross-host prediction into one accuracy across all diseases.

The page's prediction is written per-disease, and for a reason: the interesting result is that *some diseases transfer and some don't*. A single overall number averages those apart. If three diseases transfer perfectly and one collapses, the overall accuracy lands somewhere in the middle and tells you nothing about either group — it hides exactly the variation the experiment exists to find. So per-disease is the default and the level your headline finding should live at.

Reporting `"both"` is fine and often useful: per-disease as the headline, overall as a one-line summary for context. What you should not do is let the overall number *be* the finding. Lock the level here so the gap table in NB02 is built to it from the start, rather than aggregated one way and then re-cut later.

In [ ]:
### 5.1) Commit Precommit 3: cross-host accuracy aggregation
# Default is "per_disease" — the level the page's prediction is written against.
# "both" is fine if per-disease stays the headline and overall is context.
precommit["cross_host_aggregation"] = {
    "level": "per_disease",   # one of: "per_disease", "overall", "both"
    "reasoning": (
        # Replace this placeholder with your own reasoning before submission.
        # Cover at minimum: why per-disease is the reporting level the page's
        # prediction is written against; what a single overall number would
        # hide; and, if you chose "both", which one is the headline and which
        # is the secondary summary.
        "PLACEHOLDER — REWRITE BEFORE SUBMISSION."
    ),
}

# Print back what just got committed.
ag = precommit["cross_host_aggregation"]
print(f"Cross-host aggregation level: {ag['level']}")

### 6) Closeout: write `precommit.json`; submit EQ#1

Week 1 is done once this notebook has written your three decisions to `precommit.json`. The first cell below writes the file and reads it straight back to confirm it round-trips — if the write succeeded, the file on disk matches the dict you built section by section.

One thing before you submit: the three `reasoning` fields are still placeholders. The decisions themselves have sensible defaults you may have kept, but the reasoning is yours — it's the part of EQ#1 that shows you understand *why* each decision is what it is, and for the disease list especially it's where you cite the counts that justify your cut. The summary cell lists exactly which reasoning fields still need rewriting. The notebook won't stop you from submitting with placeholders left in, but a `precommit.json` full of "PLACEHOLDER" is not a finished Week-1 deliverable.

When you open NB01 next week, the first thing it does is read this file back. Anything you change between now and then changes your methodology, not just your result.

In [ ]:
### 6.1) Write precommit.json and confirm it round-trips
import json

precommit_path = OUTPUT_DIR / "precommit.json"
with open(precommit_path, "w") as f:
    json.dump(precommit, f, indent=2)

print(f"Wrote {precommit_path}")

# Sanity-load: read the file back and confirm it matches the in-memory dict.
with open(precommit_path, "r") as f:
    precommit_loaded = json.load(f)

assert precommit_loaded == precommit, \
    "Round-trip mismatch — precommit.json does not match the in-memory precommit dict."
print("Sanity-load passed: precommit.json round-trips to the in-memory dict.")

In [ ]:
### 6.2) Verdict summary and remaining-placeholder check
dl = precommit["disease_list"]
ho = precommit["hold_out_scheme"]
ag = precommit["cross_host_aggregation"]

print("Your Week-1 pre-commits:")
print()
print(f"  Disease list:            {len(dl['multi_host_diseases'])} multi-host disease(s), "
      f"{len(dl['control_single_host_diseases'])} control(s), "
      f"healthy {'in' if dl['include_healthy'] else 'out'}, "
      f"min {dl['min_images_per_host']}/host")
print(f"  Hold-out scheme:         {ho['scheme']}")
print(f"  Cross-host aggregation:  {ag['level']}")
print()

# List every reasoning field still set to the placeholder.
still_placeholder = []
for key in ["disease_list", "hold_out_scheme", "cross_host_aggregation"]:
    if precommit[key].get("reasoning", "").startswith("PLACEHOLDER"):
        still_placeholder.append(f"precommit['{key}']['reasoning']")

if still_placeholder:
    print(f"Reasoning still to rewrite ({len(still_placeholder)}):")
    for path in still_placeholder:
        print(f"  - {path}")
    print()
    print("These are placeholders, not finished reasoning. Rewrite them before "
          "submitting EQ#1.")
else:
    print("All reasoning fields have been rewritten. Ready to submit.")

print()
print("Submit precommit.json and your EQ#1 orientation note as your Week-1 deliverable.")